In [ ]:
import os

# Trick to plot with tex
os.environ["LD_LIBRARY_PATH"] = ""
os.environ["CONDA_PREFIX"] = "/home/guerrini/.conda/envs/sp_validation"

import configparser
import subprocess

from getdist import plots, loadMCSamples
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import scipy.stats as stats
from IPython.display import Markdown, display
import healpy as hp
import matplotlib.scale as mscale
import matplotlib.ticker as ticker
import matplotlib.transforms as mtransforms
import seaborn as sns

plt.style.use(
    "../papers/harmonic/matplotlib_config/paper.mplstyle"
)

plt.rcParams["text.usetex"] = True

sns.set_palette("husl")


class SquareRootScale(mscale.ScaleBase):
    """
    ScaleBase class for generating square root scale.

    Usage example: axis.set_yscale('squareroot')

    """

    name = 'squareroot'

    def __init__(self, axis, **kwargs):
        mscale.ScaleBase.__init__(self, axis, **kwargs)

    def set_default_locators_and_formatters(self, axis):
        axis.set_major_locator(ticker.AutoLocator())
        axis.set_major_formatter(ticker.ScalarFormatter())
        axis.set_minor_locator(ticker.NullLocator())
        axis.set_minor_formatter(ticker.NullFormatter())

    def limit_range_for_scale(self, vmin, vmax, minpos):
        return max(0., vmin), vmax

    class SquareRootTransform(mtransforms.Transform):
        input_dims = 1
        output_dims = 1
        is_separable = True

        def transform_non_affine(self, a):
            return np.array(a)**0.5

        def inverted(self):
            return SquareRootScale.InvertedSquareRootTransform()

    class InvertedSquareRootTransform(mtransforms.Transform):
        input_dims = 1
        output_dims = 1
        is_separable = True

        def transform(self, a):
            return np.array(a)**2

        def inverted(self):
            return SquareRootScale.SquareRootTransform()

    def get_transform(self):
        return self.SquareRootTransform()

mscale.register_scale(SquareRootScale)
%matplotlib inline
# import uncertainties

plt.rc('mathtext', fontset='stix')
plt.rc('font', family='sans-serif')

g = plots.get_subplot_plotter(width_inch=30)
g.settings.axes_fontsize=30
g.settings.axes_labelsize=30
g.settings.alpha_filled_add = 0.7
g.settings.legend_fontsize = 40


#SPECIFY DATA DIRECTORY AND DESIRED CHAINS TO ANALYSE
root_dir='/n09data/guerrini/output_chains/'

catalog_version = 'SP_v1.4.6_leak_corr_cell'
catalog_version_real_space = 'SP_v1.4.6_leak_corr_A_10_80'

path_ini_files = '/home/guerrini/sp_validation/cosmo_inference/cosmosis_config/'

roots = [
    f"SP_v1.4.6_leak_corr_A_lmin=300_lmax=1600_cell",
    f"SP_v1.4.6_leak_corr_B_lmin=300_lmax=1600_cell",
    f"SP_v1.4.6_leak_corr_C_lmin=300_lmax=1600_cell",
    f"SP_v1.4.6_leak_corr_A_10_80",
    #f"SP_v1.4.6_leak_corr_A_kmax=5Mpc_cell",
    f"SP_v1.4.6_leak_corr_A_kmax=3Mpc_cell",
    f"SP_v1.4.6_leak_corr_A_kmax=1Mpc_cell",
    f"SP_v1.4.6_leak_corr_A_include_large_scales_cell",
    f"SP_v1.4.6_leak_corr_A_small_scales_cell",
    f"SP_v1.4.6_leak_corr_A_large_scales_cell",
    f"SP_v1.4.6_leak_corr_A_halofit_cell",
    f"SP_v1.4.6_leak_corr_A_HMCode_nobar_cell",
    f"SP_v1.4.6_leak_corr_A_OneCov_cell",
    f"SP_v1.4.6_A_fid_cell"
]

labels = [
    rf"UNIONS $C_\ell$, Blind A",
    rf"UNIONS $C_\ell$, Blind B",
    rf"UNIONS $C_\ell$, Blind C",
    r"UNIONS $\xi_\pm(\vartheta)$, (Goh et al., 2026)",
    #rf"$k_\mathrm{{max}}=5 h$ Mpc$^{{-1}}$, $\ell_\mathrm{{max}}=2048$",
    rf"$k_\mathrm{{max}}=3 h$ Mpc$^{{-1}}$, $\ell_\mathrm{{max}}=1800$",
    rf"$k_\mathrm{{max}}=1 h$ Mpc$^{{-1}}$, $\ell_\mathrm{{max}}=500$",
    r"Include Large Scales, $\ell_\mathrm{max}=1600$",
    f"Small Scales only",
    f"Large Scales only",
    r"Halofit",
    r"HMCode no baryons",
    f"OneCovariance only",
    f"No leakage correction"
]

bases = [
    "harmonic",
    "harmonic",
    "harmonic",
    "configuration",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic"
]


properties = {}

for i, root in enumerate(roots):
    config = configparser.ConfigParser()
    config.optionxform = str  # Preserve case sensitivity of option names
    config.read(path_ini_files+f'/cosmosis_pipeline_{root}.ini')

    try:
        lower_bound_cell_ee, upper_bound_cell_ee = map(float, config["2pt_like"]["angle_range_CELL_EE_1_1"].split())
        
        properties[root] = {
            'lower_bound_cell_ee': lower_bound_cell_ee,
            'upper_bound_cell_ee': upper_bound_cell_ee,
        }
    except KeyError:
        properties[root] = {
            'lower_bound_cell_ee': 0.0,
            'upper_bound_cell_ee': 2048.0
        }

    if bases[i] == 'configuration':
        # Also save the scale cuts in theta for xi
        add_xi_sys = config["2pt_like"]["add_xi_sys"]
        add_xi_sys = add_xi_sys == 'T'
        lower_bound_xi_plus, upper_bound_xi_plus = map(float, config["2pt_like"]["angle_range_XI_PLUS_1_1"].split())
        lower_bound_xi_minus, upper_bound_xi_minus = map(float, config["2pt_like"]["angle_range_XI_MINUS_1_1"].split())

        properties[root].update({
            'add_xi_sys': add_xi_sys,
            'lower_bound_xi_plus': lower_bound_xi_plus,
            'upper_bound_xi_plus': upper_bound_xi_plus,
            'lower_bound_xi_minus': lower_bound_xi_minus,
            'upper_bound_xi_minus': upper_bound_xi_minus
        })


print(roots)

## Retrieve the chains

In [ ]:
# MAKE PARAMNAMES FILE

for root in roots:
    with open(root_dir + '{}/samples_{}.txt'.format('/'+root ,root), "r") as file:
        params = file.readline()[1:].split('\t')[:-4]
        file.close()
 
    with open(root_dir + '{}/getdist_{}.paramnames'.format('/'+root, root), "w") as file:
        for i in range(len(params)):
            if len(params[i].split('--')) > 1:
                file.write(params[i].split('--')[1] + '\n')
            else:
                file.write(params[i].split('--')[0] + '\n')
        file.close()

In [ ]:
#READ CHAIN

chains=[]

for root in roots:

    samples = np.loadtxt(root_dir + '{}/samples_{}.txt'.format(root,root))
    print(len(samples))
    if 'nautilus' in root:
        samples = np.column_stack((np.exp(samples[:,-3]),samples[:,-1]-samples[:,-2],samples[:,0:-3]))
    else:
        samples = np.column_stack((samples[:,-1],samples[:,-3],samples[:,0:-4]))
    np.savetxt(root_dir + '{}/getdist_{}.txt'.format(root,root), samples)
    
    chain = g.samples_for_root(root_dir + '{}/getdist_{}'.format(root,root),
                               cache=False,
                               settings={'ignore_rows':0,
                                         'smooth_scale_2D':0.3,
                                         'smooth_scale_1D':0.3})

    chains.append(chain)

In [ ]:
name_list = ['OMEGA_M','ombh2','h0','n_s','SIGMA_8','s_8_input', 'logt_agn','a','m1','bias_1']
label_list = ['\Omega_m', '\omega_b h^2', 'h_0', 'n_s', '\sigma_8', 'S_8', 'log T_{AGN}', 'A_{IA}', 'm_1', '\Delta z_1']

for chain in chains:
    param_names = chain.getParamNames()
    for name, label in zip(name_list, label_list):
        param_names.parWithName(name).label = label
    

## Extract the best fit parameters

In [ ]:
best_fit = {}

for root, chain in zip(roots, chains):
    print(root)
    likestats = chain.getLikeStats()
    bestfit_idx = np.argmax(chain.loglikes)
    maxlike = chain.loglikes[bestfit_idx]
    print(f"Maximum Likelihood: {maxlike:.5g}")
    best_fit[root] = {
        'likelihood': maxlike
    }
    margestats = chain.getMargeStats()
    s8_stats = margestats.parWithName('S_8')
    sigma8_stats = margestats.parWithName('SIGMA_8')
    omegam_stats = margestats.parWithName('OMEGA_M')
    a_ia_stats = margestats.parWithName('a')

    best_fit[root].update({
        'S_8_mean': s8_stats.mean,
        'S_8_lower': s8_stats.mean - s8_stats.limits[0].lower,
        'S_8_upper': s8_stats.limits[0].upper - s8_stats.mean,
        'sigma_8_mean': sigma8_stats.mean,
        'sigma_8_lower': sigma8_stats.mean - sigma8_stats.limits[0].lower,
        'sigma_8_upper': sigma8_stats.limits[0].upper - sigma8_stats.mean,
        'omega_m_mean': omegam_stats.mean,
        'omega_m_lower': omegam_stats.mean - omegam_stats.limits[0].lower,
        'omega_m_upper': omegam_stats.limits[0].upper - omegam_stats.mean,
        'A_IA_mean': a_ia_stats.mean,
        'A_IA_lower': a_ia_stats.mean - a_ia_stats.limits[0].lower,
        'A_IA_upper': a_ia_stats.limits[0].upper - a_ia_stats.mean
    })
    try:
        t_agn_stats = margestats.parWithName('logt_agn')
        best_fit[root].update({
            'logt_agn_mean': t_agn_stats.mean,
            'logt_agn_lower': t_agn_stats.mean - t_agn_stats.limits[0].lower,
            'logt_agn_upper': t_agn_stats.limits[0].upper - t_agn_stats.mean
        })
    except:
        pass
    for i, par in enumerate(likestats.names):
        best_fit[root].update({par.name: np.average(chain.samples[:, i], weights=chain.weights)})
    

## Run `Cosmosis` in test mode to get the data vectors

In [ ]:
if not os.path.exists(path_ini_files+'/values_empty.ini'):
    content = """[cosmological_parameters]

tau          =  0.0544
w            = -1.0
mnu = 0.06
omega_k      =  0.0
wa           =  0.0

[halo_model_parameters]

[intrinsic_alignment_parameters]

[shear_calibration_parameters]

[nofz_shifts]

[psf_leakage_parameters]
"""

    with open(path_ini_files+'/values_empty.ini', 'w') as f:
        f.write(content)
        f.close()

    print('File created successfully')

In [ ]:
section_map = {
    'omch2': 'cosmological_parameters',
    'ombh2': 'cosmological_parameters',
    'h0': 'cosmological_parameters',
    'n_s': 'cosmological_parameters',
    's_8_input': 'cosmological_parameters',
    'logt_agn': 'halo_model_parameters',
    'a': 'intrinsic_alignment_parameters',
    'm1': 'shear_calibration_parameters',
    'bias_1': 'nofz_shifts',
    'alpha': 'psf_leakage_parameters',
    'beta': 'psf_leakage_parameters',
}

In [ ]:
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/home/guerrini/.conda/envs/sp_validation/lib/python3.9/site-packages/cosmosis/datablock:" + env.get("LD_LIBRARY_PATH", "")

for root in roots:
    print(root)
    config = configparser.ConfigParser()
    config.optionxform = str  # Preserve case sensitivity of option names
    config.read(path_ini_files+'/values_empty.ini')
    for param, value in best_fit[root].items():
        section = section_map.get(param)
        if section is None:
            continue
        if section not in config:
            config.add_section(section)
        config[section][param] = str(value)

    with open(path_ini_files+'/values_empty.ini', 'w') as configfile:
        config.write(configfile)

    #Modify the ini file to run in test mode at the best fit
    config = configparser.ConfigParser()
    config.optionxform = str  # Preserve case sensitivity of option names
    config.read(path_ini_files+f'/cosmosis_pipeline_{root}.ini')

    sampler = config['runtime']['sampler']
    config['runtime']['sampler'] = 'test'
    values = config['pipeline']['values']
    config['pipeline']['values'] = path_ini_files + '/values_empty.ini'

    with open(path_ini_files+f'/cosmosis_pipeline_{root}.ini', 'w') as configfile:
        config.write(configfile)

    #Run cosmosis
    result = subprocess.run(
        ['cosmosis',  'cosmosis_config/cosmosis_pipeline_{}.ini'.format(root)],
        env=env,
        capture_output=True,
        text=True
    )
    print(f"STDOUT:\n{result.stdout}")
    print(f"STDERR:\n{result.stderr}")

    #Modify the ini file to the previous one
    config['pipeline']['values'] = values
    config['runtime']['sampler'] = sampler

    with open(path_ini_files+f'/cosmosis_pipeline_{root}.ini', 'w') as configfile:
        config.write(configfile)

## Compute the $\chi^2$

In [ ]:
output_folder = '/n09data/guerrini/output_chains/'

metrics = {}

for i, root in enumerate(roots):
    print(root)

    base = bases[i]

    if base == "harmonic":
        #Remove cell from the end of root
        root_cell_removed = root.replace('_cell', '')

        lower_bound_cell_ee = properties[root]['lower_bound_cell_ee']
        upper_bound_cell_ee = properties[root]['upper_bound_cell_ee']
        print(upper_bound_cell_ee)

        #Read the results
        ell = np.loadtxt(output_folder + 'best_fit/{}/shear_cl/ell.txt'.format(root))
        shear_cl = np.loadtxt(output_folder + 'best_fit/{}/shear_cl/bin_1_1.txt'.format(root))

        #Read the data
        data = fits.open(f'data/{root_cell_removed}/cosmosis_{root}.fits')

        ell_data = data['CELL_EE'].data['ANG']
        cell_data = data['CELL_EE'].data['VALUE']

        #Load the covariance
        cov = data['COVMAT'].data
        cov_cell = cov

        #interpolate the model
        interp_cell_ee = interp1d(ell, shear_cl, kind='cubic', fill_value='extrapolate')

        cell_model = interp_cell_ee(ell_data)

        #Apply scale cuts
        mask_cell = (ell_data > lower_bound_cell_ee) & (ell_data < upper_bound_cell_ee)
        cell_data = cell_data[mask_cell]
        cell_model = cell_model[mask_cell]
        cov_cell = cov_cell[mask_cell][:, mask_cell]

        cell_chi2 = np.dot((cell_model - cell_data), np.dot(np.linalg.inv(cov_cell), (cell_model - cell_data)))
        n_dof_cell = np.sum(mask_cell)
        print(n_dof_cell)
        n_dof_cell -= 9
        p_value_cell = 1 - stats.chi2.cdf(cell_chi2, n_dof_cell)

        metrics[root] = {
            'chi2': cell_chi2,
            'n_dof': n_dof_cell,
            'p_value': p_value_cell
        }

    elif base == "configuration":

        add_xi_sys = properties[root]['add_xi_sys']
        lower_bound_xi_plus = properties[root]['lower_bound_xi_plus']
        upper_bound_xi_plus = properties[root]['upper_bound_xi_plus']
        lower_bound_xi_minus = properties[root]['lower_bound_xi_minus']
        upper_bound_xi_minus = properties[root]['upper_bound_xi_minus']

        #Read the results
        theta = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/theta.txt'.format(root))
        theta_arcmin = theta * 180 * 60 / np.pi
        shear_xi_plus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/bin_1_1.txt'.format(root))
        shear_xi_minus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_minus/bin_1_1.txt'.format(root))
        xi_sys_plus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_plus.txt'.format(root))
        xi_sys_minus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_minus.txt'.format(root))

        #Read model tau_stats
        theta_tau = np.loadtxt(output_folder + 'best_fit/{}/tau_0_plus/theta.txt'.format(root))
        theta_tau_arcmin = theta_tau * 180 * 60 / np.pi
        tau_0_model = np.loadtxt(output_folder + 'best_fit/{}/tau_0_plus/bin_1_1.txt'.format(root))
        tau_2_model = np.loadtxt(output_folder + 'best_fit/{}/tau_2_plus/bin_1_1.txt'.format(root))

        #Read the data
        data = fits.open(f'data/{catalog_version_real_space}/cosmosis_{catalog_version_real_space}.fits')

        theta_data = data['XI_PLUS'].data['ANG']
        xi_plus_data = data['XI_PLUS'].data['VALUE']
        xi_minus_data = data['XI_MINUS'].data['VALUE']
        tau_0_data = data['TAU_0_PLUS'].data['VALUE']
        tau_2_data = data['TAU_2_PLUS'].data['VALUE']

        #Load the covariance
        cov = data['COVMAT'].data
        cov_xi = cov[0:2*len(xi_plus_data), 0:2*len(xi_plus_data)]
        cov_tau = cov[2*len(xi_plus_data):, 2*len(xi_plus_data):]

        #interpolate the model
        interp_xi_plus = interp1d(theta_arcmin, shear_xi_plus, kind='cubic', fill_value='extrapolate')
        interp_xi_minus = interp1d(theta_arcmin, shear_xi_minus, kind='cubic', fill_value='extrapolate')

        xi_plus_model = interp_xi_plus(theta_data)
        if add_xi_sys:
            xi_plus_model += xi_sys_plus
        xi_minus_model = interp_xi_minus(theta_data)
        if add_xi_sys:
            xi_minus_model += xi_sys_minus

        #Concatenate the data vector
        xi_data = np.concatenate((xi_plus_data, xi_minus_data))
        xi_model = np.concatenate((xi_plus_model, xi_minus_model))

        tau_data = np.concatenate((tau_0_data, tau_2_data))
        tau_model = np.concatenate((tau_0_model, tau_2_model))

        #Apply scale cuts
        mask_xi_plus = (theta_data > lower_bound_xi_plus) & (theta_data < upper_bound_xi_plus)
        mask_xi_minus = (theta_data > lower_bound_xi_minus) & (theta_data < upper_bound_xi_minus)
        mask = np.concatenate((mask_xi_plus, mask_xi_minus))

        xi_data = xi_data[mask]
        xi_model = xi_model[mask]
        cov_xi = cov_xi[mask][:, mask]


        xi_plus_chi2 = np.dot((xi_model - xi_data), np.dot(np.linalg.inv(cov_xi), (xi_model - xi_data)))
        tau_chi2 = np.dot((tau_model - tau_data), np.dot(np.linalg.inv(cov_tau), (tau_model - tau_data)))
        n_dof_xi = np.sum(mask)
        n_dof_xi -= 11
        n_dof_tau = len(tau_0_data) + len(tau_2_data)
        p_value_xi = 1 - stats.chi2.cdf(xi_plus_chi2, n_dof_xi)
        p_value_tau = 1 - stats.chi2.cdf(tau_chi2, n_dof_tau)
        chi2_tot = xi_plus_chi2 + tau_chi2
        n_dof_tot = n_dof_xi + n_dof_tau
        p_value_tot = 1 - stats.chi2.cdf(chi2_tot, n_dof_tot)

        metrics[root] = {
            'chi2': xi_plus_chi2,
            'n_dof': n_dof_xi,
            'p_value': p_value_xi
        }

    
    print("Done!")

In [ ]:
def get_latex_table(metrics):
    latex_lines = [
        r"\begin{tabular}{|l|c|c|c|c|c|c|c|}",
        r"\hline",
        r"Experiment name & $S_8$ & $\Omega_m$ & $\sigma_8$ & $A_\mathrm{IA}$ & $\log T_\mathrm{AGN}$ & $\chi^2$/dof & PTE \\ ",
        r"\hline"
    ]

    for i, (root, vals) in enumerate(metrics.items()):
        label = labels[i]
        best_fit_vals = best_fit[root]
        log_t_agn_mean = best_fit_vals.get('logt_agn_mean', None)

        if log_t_agn_mean is None:
            logt_agn_mean_str = "N/A"
        else:
            logt_agn_mean_str = f"${best_fit_vals['logt_agn_mean']:.3f}^{{+{best_fit_vals['logt_agn_upper']:.3f}}}_{{-{best_fit_vals['logt_agn_lower']:.3f}}}$"
        line = (
            f"{label} & "
            rf" ${best_fit_vals['S_8_mean']:.3f}^{{+{best_fit_vals['S_8_upper']:.3f}}}_{{-{best_fit_vals['S_8_lower']:.3f}}}$ & "
            rf" ${best_fit_vals['omega_m_mean']:.3f}^{{+{best_fit_vals['omega_m_upper']:.3f}}}_{{-{best_fit_vals['omega_m_lower']:.3f}}}$ & "
            rf" ${best_fit_vals['sigma_8_mean']:.3f}^{{+{best_fit_vals['sigma_8_upper']:.3f}}}_{{-{best_fit_vals['sigma_8_lower']:.3f}}}$ & "
            rf" ${best_fit_vals['A_IA_mean']:.3f}^{{+{best_fit_vals['A_IA_upper']:.3f}}}_{{-{best_fit_vals['A_IA_lower']:.3f}}}$ & "
            rf" {logt_agn_mean_str} & "
            f"{vals['chi2']:.2f}/{vals['n_dof']} & {vals['p_value']:.5f} \\\\"
        )
        latex_lines.append(line)

    latex_lines.append(r"\hline")
    latex_lines.append(r"\end{tabular}")

    # Print LaTeX table
    print("\n".join(latex_lines))

In [ ]:
get_latex_table(metrics)

In [ ]:
def display_markdown(metrics):
    # Build Markdown table
    header = (
        "| Root | $\chi^2$ ($C_\ell$) / dof | p-val ($C_\ell$) |\n"
        "|------|----------------|------------|\n"
    )

    rows = []
    for root, vals in metrics.items():
        row = f"| `{root}` "
        row += f"| {vals['chi2']:.2f} / {vals['n_dof']} "
        row += f"| {vals['p_value']:.5f} "
        rows.append(row)

    # Display in Jupyter
    display(Markdown(header + "\n".join(rows)))
    return header + "\n".join(rows)

In [ ]:
markdown_source = display_markdown(metrics)

In [ ]:
markdown_source

## Plot the best-fit of each model

In [ ]:
catalog_version = 'SP_v1.4.6_leak_corr_A_lmin=300_lmax=1600'
data = fits.open(f'/home/guerrini/sp_validation/cosmo_inference/data/{catalog_version}/cosmosis_{catalog_version}_cell.fits')
cell_ee = data['CELL_EE'].data
cov_mat = data['COVMAT'].data

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 8))

ell = cell_ee['ANG']
cell = cell_ee['VALUE']

ax.errorbar(ell, ell*cell, yerr=ell*np.sqrt(np.diag(cov)), fmt='o', label="SP_v1.4.5 data", color='black')
ax.set_xlabel('$\ell$')
ax.set_ylabel('$\ell C_\ell$')
ax.set_xlim(ell.min()-10, ell.max()+100)
ax.set_xscale('squareroot')
ax.set_xticks(np.array([100, 400, 900, 1600]))
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", length=2, width=0.8)
minor_ticks = [i*10 for i in range(1, 10)] + [i*100 for i in range(1, 21)]
ax.xaxis.set_ticks(minor_ticks, minor=True)

plt.legend(fontsize=15)

plt.show()

In [ ]:
def plot_best_fit(data_points, root_to_plot, line_args, savefile, ell_min=10.0, ell_max=2048.0, multiply_ell=True, loc_legend="best", bbox_to_anchor=None, label_data="Fiducial data", labels=None):
    data = fits.open(f'/home/guerrini/sp_validation/cosmo_inference/data/{data_points}/cosmosis_{data_points}_cell.fits')
    cell_ee = data['CELL_EE'].data
    cov_mat = data['COVMAT'].data

    if labels is None:
        labels = root_to_plot

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    ell, cell = cell_ee['ANG'], cell_ee['VALUE']
    ax.errorbar(ell, ell*cell, yerr=ell*np.sqrt(np.diag(cov_mat)), fmt='o', label=label_data, color='black', capsize=2)
    
    for idx, (label, root) in enumerate(zip(labels, root_to_plot)):
        lower_bound_cell_ee = properties[root]['lower_bound_cell_ee']
        upper_bound_cell_ee = properties[root]['upper_bound_cell_ee']
        
        #Read the results
        ell = np.loadtxt(output_folder + 'best_fit/{}/shear_cl/ell.txt'.format(root))
        shear_cl = np.loadtxt(output_folder + 'best_fit/{}/shear_cl/bin_1_1.txt'.format(root))

        mask = (ell > ell_min) & (ell < ell_max)

        ax.plot(ell[mask], ell[mask]*shear_cl[mask] if multiply_ell else shear_cl[mask], label=label, **line_args[idx])
    
    # Plot the scale cuts for different k_max
    ax.axvline(x=1800, color='black', linestyle='--', alpha=0.5)
    ax.axvline(x=2048, color='black', linestyle='--', alpha=1.0)
    ax.axvline(x=500, color='black', linestyle='--', alpha=0.3)

    
    

    # Add labels directly under the tick
    ax.text(1740, 0.90,
            r"$k_\mathrm{max} = 3 h$ Mpc$^{-1}$",
            transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=14, rotation=90)

    ax.text(1978, 0.90,
            r"$k_\mathrm{max} = 5 h$ Mpc$^{-1}$",
            transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=14, rotation=90)

    ax.text(470,  0.90,
            r"$k_\mathrm{max} = 1 h$ Mpc$^{-1}$",
            transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=14, rotation=90)

    ell, cell = cell_ee['ANG'], cell_ee['VALUE']
    ax.set_ylabel(r'$\ell C_\ell \times 10^{-7}$', fontsize=20)
    ax.set_xlabel(r'Multipole $\ell$', fontsize=20)
    ax.set_xlim(ell.min()-10, ell.max()+100)
    ax.set_xscale('squareroot')
    ax.set_xticks(np.array([100, 400, 900, 1600]))
    ax.minorticks_on()
    ax.tick_params(axis="x", which="minor", length=2, width=0.8)
    minor_ticks = [i*10 for i in range(1, 10)] + [i*100 for i in range(1, 21)]
    ax.xaxis.set_ticks(minor_ticks, minor=True)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.tick_params(axis='both', which='minor', labelsize=10)
    ax.yaxis.get_offset_text().set_visible(False)


    plt.legend(loc=loc_legend, bbox_to_anchor=bbox_to_anchor, fontsize=11)

    if savefile is not None:
        plt.savefig(savefile, bbox_inches='tight')

    plt.show()

def plot_best_fit_ratio(root_to_plot, colours, savefile, theta_min=1.0, theta_max=250.0):
    data = fits.open(f'/home/guerrini/sp_validation/cosmo_inference/data/{catalog_version}/cosmosis_{catalog_version}.fits')
    xi_plus = data['XI_PLUS'].data
    xi_minus = data['XI_MINUS'].data
    cov_mat = data['COVMAT'].data

    plt.figure(figsize=(15, 15))

    plt.subplot(211)

    root = roots[0]
    add_xi_sys = properties[root]['add_xi_sys']
    lower_bound_xi_plus = properties[root]['lower_bound_xi_plus']
    upper_bound_xi_plus = properties[root]['upper_bound_xi_plus']
    lower_bound_xi_minus = properties[root]['lower_bound_xi_minus']
    upper_bound_xi_minus = properties[root]['upper_bound_xi_minus']

    #Read the results
    theta = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/theta.txt'.format(root))
    theta_arcmin = theta * 180 * 60 / np.pi
    shear_xi_plus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/bin_1_1.txt'.format(root))
    shear_xi_minus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_minus/bin_1_1.txt'.format(root))
    xi_sys_plus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_plus.txt'.format(root))
    xi_sys_minus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_minus.txt'.format(root))
    theta_xi_sys = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/theta.txt'.format(root))
    theta_xi_sys_arcmin = theta_xi_sys * 180 * 60 / np.pi

    mask = (theta_arcmin > theta_min) & (theta_arcmin < theta_max)
    xi_plus_model_fiducial = shear_xi_plus[mask]
    if add_xi_sys:
        xi_plus_model_fiducial += np.interp(theta_arcmin[mask], theta_xi_sys_arcmin, xi_sys_plus)

    plt.errorbar(xi_plus['ANG'], xi_plus['VALUE']/np.interp(xi_plus['ANG'], theta_arcmin[mask], xi_plus_model_fiducial), yerr=np.sqrt(np.diag(cov_mat))[:20]/np.abs(np.interp(xi_plus['ANG'], theta_arcmin[mask], xi_plus_model_fiducial)), fmt='o', label=f'{catalog_version} data', color='black', markersize=2)

    for root, color in zip(root_to_plot, colours):
        add_xi_sys = properties[root]['add_xi_sys']
        lower_bound_xi_plus = properties[root]['lower_bound_xi_plus']
        upper_bound_xi_plus = properties[root]['upper_bound_xi_plus']
        lower_bound_xi_minus = properties[root]['lower_bound_xi_minus']
        upper_bound_xi_minus = properties[root]['upper_bound_xi_minus']

        #Read the results
        theta = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/theta.txt'.format(root))
        theta_arcmin = theta * 180 * 60 / np.pi
        shear_xi_plus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/bin_1_1.txt'.format(root))
        shear_xi_minus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_minus/bin_1_1.txt'.format(root))
        xi_sys_plus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_plus.txt'.format(root))
        xi_sys_minus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_minus.txt'.format(root))
        theta_xi_sys = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/theta.txt'.format(root))
        theta_xi_sys_arcmin = theta_xi_sys * 180 * 60 / np.pi

        mask = (theta_arcmin > theta_min) & (theta_arcmin < theta_max)
        xi_plus_model = shear_xi_plus[mask]
        if add_xi_sys:
            xi_plus_model += np.interp(theta_arcmin[mask], theta_xi_sys_arcmin, xi_sys_plus)
        
        alpha = 1.0 if root==roots[0] else 0.5
        plt.plot(theta_arcmin[mask], xi_plus_model/xi_plus_model_fiducial, color=color, label=root, alpha=alpha)
        plt.axvline(x=lower_bound_xi_plus, color=color, linestyle='--', alpha=0.3)
        plt.axvline(x=upper_bound_xi_plus, color=color, linestyle='--', alpha=0.3)
        


    plt.ylabel(fr'$\xi_{{+}}/\xi_{{+, \text{{fid}}}}$', fontsize=26)
    plt.xscale('log')
    #plt.yscale('log')
    plt.legend(loc="lower left", fontsize=8)

    plt.subplot(212)

    root = roots[0]
    add_xi_sys = properties[root]['add_xi_sys']
    lower_bound_xi_plus = properties[root]['lower_bound_xi_plus']
    upper_bound_xi_plus = properties[root]['upper_bound_xi_plus']
    lower_bound_xi_minus = properties[root]['lower_bound_xi_minus']
    upper_bound_xi_minus = properties[root]['upper_bound_xi_minus']

    #Read the results
    theta = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/theta.txt'.format(root))
    theta_arcmin = theta * 180 * 60 / np.pi
    shear_xi_plus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/bin_1_1.txt'.format(root))
    shear_xi_minus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_minus/bin_1_1.txt'.format(root))
    xi_sys_plus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_plus.txt'.format(root))
    xi_sys_minus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_minus.txt'.format(root))
    theta_xi_sys = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/theta.txt'.format(root))
    theta_xi_sys_arcmin = theta_xi_sys * 180 * 60 / np.pi

    mask = (theta_arcmin > theta_min) & (theta_arcmin < theta_max)
    xi_minus_model_fiducial = shear_xi_minus[mask]
    if add_xi_sys:
        xi_minus_model_fiducial += np.interp(theta_arcmin[mask], theta_xi_sys_arcmin, xi_sys_minus)

    plt.errorbar(xi_minus['ANG'], xi_minus['VALUE']/np.interp(xi_minus['ANG'], theta_arcmin[mask], xi_minus_model_fiducial), yerr=np.sqrt(np.diag(cov_mat))[20:40]/np.abs(np.interp(xi_minus['ANG'], theta_arcmin[mask], xi_minus_model_fiducial)), fmt='o', label=f'{catalog_version} data', color='black', markersize=2)

    for root, color in zip(root_to_plot, colours):
        add_xi_sys = properties[root]['add_xi_sys']
        lower_bound_xi_plus = properties[root]['lower_bound_xi_plus']
        upper_bound_xi_plus = properties[root]['upper_bound_xi_plus']
        lower_bound_xi_minus = properties[root]['lower_bound_xi_minus']
        upper_bound_xi_minus = properties[root]['upper_bound_xi_minus']

        #Read the results
        theta = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/theta.txt'.format(root))
        theta_arcmin = theta * 180 * 60 / np.pi
        shear_xi_plus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_plus/bin_1_1.txt'.format(root))
        shear_xi_minus = np.loadtxt(output_folder + 'best_fit/{}/shear_xi_minus/bin_1_1.txt'.format(root))
        xi_sys_plus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_plus.txt'.format(root))
        xi_sys_minus = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/shear_xi_minus.txt'.format(root))
        theta_xi_sys = np.loadtxt(output_folder + 'best_fit/{}/xi_sys/theta.txt'.format(root))
        theta_xi_sys_arcmin = theta_xi_sys * 180 * 60 / np.pi

        mask = (theta_arcmin > theta_min) & (theta_arcmin < theta_max)
        xi_minus_model = shear_xi_minus[mask]
        if add_xi_sys:
            xi_minus_model += np.interp(theta_arcmin[mask], theta_xi_sys_arcmin, xi_sys_minus)

        alpha = 1.0 if root==roots[0] else 0.5
        plt.plot(theta_arcmin[mask], xi_minus_model/xi_minus_model_fiducial, color=color, label=root, alpha=alpha)
        plt.axvline(x=lower_bound_xi_minus, color=color, linestyle='--', alpha=0.3)
        plt.axvline(x=upper_bound_xi_minus, color=color, linestyle='--', alpha=0.3)


    plt.xlabel(r'$\theta$ [arcmin]', fontsize=26)
    plt.ylabel(fr'$\xi_{{-}}/\xi_{{-, \text{{fid}}}}$', fontsize=26)
    plt.xscale('log')
    plt.ylim(0, 2)
    #plt.yscale('log')
    plt.legend(loc="lower left", fontsize=8)

    if savefile is not None:
        plt.savefig(savefile, bbox_inches='tight')

    plt.show()

In [ ]:
plt.rcParams.update({'font.family': "serif"})

root_to_plot = [
    f"SP_v1.4.6_leak_corr_A_lmin=300_lmax=1600_cell",
    f"SP_v1.4.6_leak_corr_A_halofit_cell",
    f"SP_v1.4.6_leak_corr_A_10_80"
]

labels = [
    r"UNIONS $C_\ell$, Blind A",
    r"UNIONS $C_\ell$, Halofit",
    r"UNIONS $\xi_\pm(\vartheta)$ (Goh et al., 2026)"
]

line_args = [
    {'color': 'royalblue', 'linestyle': '-'},
    {'color': 'royalblue', 'linestyle': '--'},
    {'color': 'orange', 'linestyle': '-'}
]

log_legend="lower center"
bbox_to_anchor=(0.685, 0.70)

savefile = "../papers/harmonic/plots/paperplot_Cell_EE_and_best_fit.png"

plot_best_fit(catalog_version, root_to_plot, line_args, savefile, labels=labels, loc_legend=log_legend, bbox_to_anchor=bbox_to_anchor)

savefile = "../papers/harmonic/plots/paperplot_Cell_EE_and_best_fit.pdf"

plot_best_fit(catalog_version, root_to_plot, line_args, savefile, labels=labels, loc_legend=log_legend, bbox_to_anchor=bbox_to_anchor)

In [ ]:
root_to_plot = [
    f"SP_v1.4.6_leak_corr_A_small_scales_cell",
    f"SP_v1.4.6_leak_corr_A_large_scales_cell"
]

labels = [
    r"Small scales only",
    r"Large scales only"
]

line_args = [
    {'color': 'royalblue', 'linestyle': '-'},
    {'color': 'royalblue', 'linestyle': '--'}
]

savefile = "../papers/harmonic/plots/small_vs_large_scale.png"

plot_best_fit(catalog_version, root_to_plot, line_args, savefile, labels=labels)

In [ ]:
# DEPRECATED code

root_to_plot = [
    "SP_v1.4.5_A",
    #"SP_v1.4.5_A_no_IA",
    #"SP_v1.4.5_A_no_dz",
    #"SP_v1.4.5_A_no_m_bias",
    "SP_v1.4.5_A_sc_3_150",
    "SP_v1.4.5_A_sc_3_60",
    "SP_v1.4.5_A_sc_10_150",
    "SP_v1.4.5_A_sc_10_60",
    "SP_v1.4.5_A_sc_5_150",
    "SP_v1.4.5_A_sc_7_150",
    #"SP_v1.4.5_A_no_leakage"
]

""" root_to_plot = [
    f"SP_v1.4.5_leak_corr_A_minsep=1_maxsep=250_nbins=20_npatch=1_sc_{int(i)}.0_80.0_10.0_80.0" for i in [3, 5, 7, 10, 11]
] """

root_to_plot = [
    "SP_v1.4.5_leak_corr_A_minsep=1_maxsep=250_nbins=20_npatch=1_sc_10.0_80.0_10.0_80.0",
    "SP_v1.4.5_leak_corr_A_minsep=1_maxsep=250_nbins=20_npatch=1_sc_10.0_80.0_10.0_80.0_no_alpha_beta"
]


colours = [
    'red', 'salmon', 'darkorange', 'forestgreen', 'turquoise', 'darkviolet', 'crimson', 'gold', 'lightcoral', 'mediumseagreen', 'lightsteelblue', 'black', 'silver', 'peru', 'maroon', 'olive'
]

savefile = 'best_fit_ratio_w_wo_leakage.png'

plot_best_fit_ratio(root_to_plot, colours, savefile)

In [ ]:
def plot_best_fit_tau(root_to_plot, colours, savefile, theta_min=1.0, theta_max=250.0):
    data = fits.open(f'/home/guerrini/sp_validation/cosmo_inference/data/{catalog_version}/cosmosis_{catalog_version}.fits')
    tau_0 = data['TAU_0_PLUS'].data
    tau_2 = data['TAU_2_PLUS'].data
    cov_mat = data['COVMAT'].data

    plt.figure(figsize=(15, 15))

    plt.subplot(211)

    plt.errorbar(tau_0['ANG'], tau_0['VALUE'], yerr=np.sqrt(np.diag(cov_mat))[40:60], fmt='o', label=f'{catalog_version} data', color='black', markersize=2)

    for root, color in zip(root_to_plot, colours):

        #Read the results
        theta = np.loadtxt(output_folder + 'best_fit/{}/tau_0_plus/theta.txt'.format(root))
        theta_arcmin = theta * 180 * 60 / np.pi
        tau_0_plus = np.loadtxt(output_folder + 'best_fit/{}/tau_0_plus/bin_1_1.txt'.format(root))


        mask = (theta_arcmin > theta_min) & (theta_arcmin < theta_max)
        
        plt.plot(theta_arcmin[mask], tau_0_plus[mask], color=color, label=root, alpha=0.5)
        
    plt.ylabel(r'$\tau_0$', fontsize=26)
    plt.xscale('log')
    #plt.yscale('log')
    plt.legend(loc="upper right", fontsize=8)

    plt.subplot(212)

    y_plot_tau_2 = tau_2['ANG'] * tau_2['VALUE']
    y_errorbar = tau_2['ANG'] * np.sqrt(np.diag(cov_mat))[60:80]
    plt.errorbar(tau_2['ANG'], y_plot_tau_2, yerr=y_errorbar, fmt='o', label=f'{catalog_version} data', color='black', markersize=2)

    for root, color in zip(root_to_plot, colours):
        #Read the results
        theta = np.loadtxt(output_folder + 'best_fit/{}/tau_2_plus/theta.txt'.format(root))
        theta_arcmin = theta * 180 * 60 / np.pi
        tau_2_plus = np.loadtxt(output_folder + 'best_fit/{}/tau_2_plus/bin_1_1.txt'.format(root))

        mask = (theta_arcmin > theta_min) & (theta_arcmin < theta_max)
        
        plt.plot(theta_arcmin[mask], theta_arcmin[mask]*tau_2_plus[mask], color=color, label=root, alpha=0.5)


    plt.xlabel(r'$\theta$ [arcmin]', fontsize=26)
    plt.ylabel(r'$\theta \tau_2$', fontsize=26)
    plt.xscale('log')
    #plt.yscale('log')
    plt.legend(loc="upper left", fontsize=8)

    if savefile is not None:
        plt.savefig(savefile, bbox_inches='tight')

    plt.show()

In [ ]:
root_to_plot = [
    "SP_v1.4.5_A",
    #"SP_v1.4.5_A_no_IA",
    #"SP_v1.4.5_A_no_dz",
    #"SP_v1.4.5_A_no_m_bias",
    "SP_v1.4.5_A_sc_3_150",
    "SP_v1.4.5_A_sc_3_60",
    "SP_v1.4.5_A_sc_10_150",
    "SP_v1.4.5_A_sc_10_60",
    "SP_v1.4.5_A_sc_5_150",
    "SP_v1.4.5_A_sc_7_150",
    #"SP_v1.4.5_A_no_leakage"
]

root_to_plot = [
    f"SP_v1.4.5_leak_corr_A_minsep=1_maxsep=250_nbins=20_npatch=1_sc_{int(i)}.0_80.0_10.0_80.0" for i in [3, 5, 7, 10, 11]
]

colours = [
    'red', 'salmon', 'darkorange', 'forestgreen', 'turquoise', 'darkviolet', 'crimson', 'gold', 'lightcoral', 'mediumseagreen', 'lightsteelblue', 'black', 'silver', 'peru', 'maroon', 'olive'
]

savefile = 'best_fit_tau_new_binning.png'

plot_best_fit_tau(root_to_plot, colours, savefile)

In [ ]:
pseudo_cell = fits.open('/home/guerrini/sp_validation/cosmo_val/output/pseudo_cl_SP_v1.4.5.fits')[1].data
cov_pseudo_cell = fits.open('/home/guerrini/sp_validation/cosmo_val/output/pseudo_cl_cov_SP_v1.4.5.fits')

theory_ell = np.loadtxt('/n09data/guerrini/output_chains/best_fit/SP_v1.4.5_A/shear_cl/ell.txt')
theory_cell = np.loadtxt('/n09data/guerrini/output_chains/best_fit/SP_v1.4.5_A/shear_cl/bin_1_1.txt')

pw = hp.pixwin(1024, lmax=2048)

plt.errorbar(pseudo_cell['ELL'], pseudo_cell['ELL']*pseudo_cell['EE'], yerr=pseudo_cell['ELL']*np.sqrt(np.diag(cov_pseudo_cell['COVAR_EE_EE'].data)), capsize=2, c='k', fmt='o', markersize=2)

mask = (theory_ell > 0.1) & (theory_ell < 2048)
plt.plot(theory_ell[mask], theory_ell[mask]*theory_cell[mask]*np.interp(theory_ell[mask], np.arange(0, 2049), pw)**2, c='r', label='best-fit $\\theta \in [3-200]$')

plt.xlabel(r'$\ell$', fontsize=26)
plt.ylabel(r'$\ell C_\ell^{EE}$', fontsize=26)
plt.legend()
plt.savefig("SP_v1.4.5_A_cell.png")
plt.show()

In [ ]:
cov_pseudo_cell.info()